# Merging Tables

Consolidates the results in `src/Relatorios_Finais/`. Each method folder (for example `VNS-Referencia`) contains one CSV per test, named `Resumo_Geral_..._Teste_<n>.csv`, with the columns `Mina;Fo;Tempo(s);Iter`.

For each method folder, the notebook writes `<nome_metodo>.csv` inside that same folder, with one row per mine.

Output format: `Mina;fo_1;tempo_1;...;fo_N;tempo_N;iter`, separator `;`, times with a decimal comma. Cells without a value (mine missing from a test) are left empty.

In [ ]:
# Consolidation of the test results in Relatorios_Finais.
# Each method folder holds one run file per test, named ..._Teste_<n>.csv, with the columns Mina;Fo;Tempo(s);Iter.
import os
import re
from pathlib import Path

import pandas as pd

# Folder that holds one sub-folder per method. The first candidate matches running this notebook
# from src/Tratamento_Resultados/; the others cover running from src/ or from the repository root.
CANDIDATAS = [
    Path('..') / 'Relatorios_Finais',
    Path('Relatorios_Finais'),
    Path('src') / 'Relatorios_Finais',
]
caminho_relatorios = next((p for p in CANDIDATAS if p.is_dir()), None)
if caminho_relatorios is None:
    raise FileNotFoundError('Pasta Relatorios_Finais nao encontrada a partir do diretorio atual.')

# Matches the run file names written by main.cpp and captures the test number, e.g. "..._Teste_12.csv" -> 12.
PADRAO_TESTE = re.compile(r'_Teste_(\d+)\.csv$')


# Combines all run files of one method folder into one table with one row per mine.
# Writes <nome_metodo>.csv in that folder and returns the numeric DataFrame (None if there are no run files).
def combinar_testes(pasta_metodo, nome_metodo):
    # Map test number -> full path of its run file.
    arquivos = {}
    for nome in os.listdir(pasta_metodo):
        casamento = PADRAO_TESTE.search(nome)
        if casamento:
            arquivos[int(casamento.group(1))] = os.path.join(pasta_metodo, nome)

    if not arquivos:
        print(f"Aviso: nenhum arquivo de teste em {pasta_metodo}")
        return None

    # Test numbers in increasing order; they become the fo_<n> and tempo_<n> columns.
    numeros_teste = sorted(arquivos)

    # Accumulator per mine: mine number -> {'fo': {test: Fo}, 'tempo': {test: time}, 'iter': iteration count}.
    dados_minas = {}
    for n in numeros_teste:
        df = pd.read_csv(arquivos[n], sep=';')
        for _, row in df.iterrows():
            mina = int(row['Mina'])
            # The first time a mine appears, its entry is created and its iteration count is taken from that run.
            if mina not in dados_minas:
                dados_minas[mina] = {'fo': {}, 'tempo': {}, 'iter': int(row['Iter'])}
            dados_minas[mina]['fo'][n] = int(row['Fo'])
            dados_minas[mina]['tempo'][n] = float(row['Tempo(s)'])

    # One output row per mine, in increasing mine number.
    linhas = []
    for mina in sorted(dados_minas):
        linha = {'Mina': mina}
        for n in numeros_teste:
            # A mine missing from a test gets None for that test, which is written as an empty cell.
            linha[f'fo_{n}'] = dados_minas[mina]['fo'].get(n)
            linha[f'tempo_{n}'] = dados_minas[mina]['tempo'].get(n)
        linha['iter'] = dados_minas[mina]['iter']
        linhas.append(linha)

    df_final = pd.DataFrame(linhas)

    # Types: nullable integers for Mina, Fo and iter (so missing values survive); float for the times.
    tipos = {'Mina': 'Int64', 'iter': 'Int64'}
    for n in numeros_teste:
        tipos[f'fo_{n}'] = 'Int64'
        tipos[f'tempo_{n}'] = 'float64'
    df_final = df_final.astype(tipos)

    # Count of missing Fo values (mines absent from some tests), reported for checking.
    colunas_fo = [f'fo_{n}' for n in numeros_teste]
    faltantes = int(df_final[colunas_fo].isna().sum().sum())

    # Copy used for saving, so that df_final keeps numeric times for the return value.
    # Times are written with ',' as the decimal mark; missing times stay empty.
    df_salvar = df_final.copy()
    for n in numeros_teste:
        coluna = f'tempo_{n}'
        df_salvar[coluna] = df_salvar[coluna].map(lambda x: '' if pd.isna(x) else str(x).replace('.', ','))

    caminho_saida = os.path.join(pasta_metodo, f'{nome_metodo}.csv')
    df_salvar.to_csv(caminho_saida, sep=';', index=False)

    print(f"{nome_metodo}: {len(numeros_teste)} testes, {len(df_final)} minas, {faltantes} valores de Fo ausentes")
    print(f"  salvo em: {caminho_saida}")
    return df_final


In [ ]:
# Process every method folder found in Relatorios_Finais, in alphabetical order.
print(f"Pasta de entrada: {caminho_relatorios.resolve()}\n")
resultados = {}
for pasta in sorted(p for p in caminho_relatorios.iterdir() if p.is_dir()):
    resultados[pasta.name] = combinar_testes(str(pasta), pasta.name)

print("\nConcluido.")
